In [4]:
import json
import pandas as pd
import duckdb
import numpy as np
import glob
from pathlib import Path

# Lendo dados

In [42]:
caminhos_arquivos = glob.glob('dados_brutos/professores/ufrj/*.json')

In [43]:
# 1. LISTAS PARA ACUMULAR OS DADOS
# ---------------------------------------------------------
# Dados Gerais
lista_pessoas = []
lista_bancas = []
lista_eventos = []
lista_orientacoes = []
lista_premios = []
lista_projetos = []

# Produção Bibliográfica
lista_bib_artigos = []
lista_bib_livros = []
lista_bib_capitulos = []
lista_bib_trabalhos_congresso = []
lista_bib_resumos_expandidos = []
lista_bib_resumos_congresso = []
lista_bib_artigos_aceitos = []
lista_bib_apresentacoes = []
lista_bib_textos_jornais = []
lista_bib_outras = []

# Produção Técnica
lista_tec_softwares_patente = []
lista_tec_softwares_sem_patente = []
lista_tec_produtos = []
lista_tec_processos = []
lista_tec_trabalhos = []
lista_tec_outras = []
lista_tec_entrevistas = []

# Patentes e Registros
lista_pat_patentes = []
lista_pat_programas = []
lista_pat_desenhos = []

print("Iniciando o processamento dos arquivos JSON...")
#caminhos_arquivos = glob.glob('dados_brutos/*.json')

if not caminhos_arquivos:
    print("ERRO: Nenhum arquivo JSON encontrado na pasta 'dados_brutos/'.")
    exit()

# ---------------------------------------------------------
# 2. EXTRAÇÃO E ACHATAMENTO (FLATTEN)
# ---------------------------------------------------------
for arquivo in caminhos_arquivos:
    with open(arquivo, 'r', encoding='utf-8') as f:
        dados = json.load(f)
        
        id_lattes = dados.get('informacoes_pessoais', {}).get('id_lattes')
        if not id_lattes:
            continue
            
        # --- PESSOAS ---
        df_pessoa = pd.json_normalize(dados['informacoes_pessoais'])
        lista_pessoas.append(df_pessoa)
        
        # --- BANCAS ---
        if 'bancas' in dados:
            for categoria, itens in dados['bancas'].items():
                if itens:
                    df_temp = pd.DataFrame(itens)
                    df_temp['id_lattes'] = id_lattes
                    df_temp['categoria_banca'] = categoria 
                    if 'membros_banca' in df_temp.columns:
                        df_temp['membros_banca'] = df_temp['membros_banca'].astype(str)
                    lista_bancas.append(df_temp)

        # --- EVENTOS ---
        if 'eventos' in dados:
            for categoria, itens in dados['eventos'].items():
                if itens:
                    df_temp = pd.DataFrame(itens)
                    df_temp['id_lattes'] = id_lattes
                    df_temp['categoria_evento'] = categoria
                    lista_eventos.append(df_temp)

        # --- ORIENTAÇÕES ---
        if 'orientacoes' in dados:
            for status, dicionario_niveis in dados['orientacoes'].items():
                for nivel, itens in dicionario_niveis.items():
                    if itens:
                        df_temp = pd.DataFrame(itens)
                        df_temp['id_lattes'] = id_lattes
                        df_temp['status'] = status
                        df_temp['nivel'] = nivel   
                        lista_orientacoes.append(df_temp)

        # --- PRÊMIOS ---
        if 'premios_titulos' in dados and dados['premios_titulos']:
            df_temp = pd.DataFrame(dados['premios_titulos'])
            df_temp['id_lattes'] = id_lattes
            lista_premios.append(df_temp)

        # --- PROJETOS ---
        if 'projetos_pesquisa' in dados and dados['projetos_pesquisa']:
            df_temp = pd.DataFrame(dados['projetos_pesquisa'])
            df_temp['id_lattes'] = id_lattes
            for col in ['descricao', 'integrantes', 'financiadores']:
                if col in df_temp.columns:
                    df_temp[col] = df_temp[col].astype(str)
            lista_projetos.append(df_temp)

        # --- PRODUÇÃO BIBLIOGRÁFICA ---
        prod_bib = dados.get('producao_bibliografica', {})
        def add_to_list(chave, lista_destino):
            itens = prod_bib.get(chave, [])
            if itens:
                df_temp = pd.DataFrame(itens)
                df_temp['id_lattes'] = id_lattes
                lista_destino.append(df_temp)

        add_to_list('artigos_periodicos', lista_bib_artigos)
        add_to_list('livros_publicados', lista_bib_livros)
        add_to_list('capitulos_livros', lista_bib_capitulos)
        add_to_list('trabalhos_completos_congressos', lista_bib_trabalhos_congresso)
        add_to_list('resumos_expandidos', lista_bib_resumos_expandidos)
        add_to_list('resumos_congressos', lista_bib_resumos_congresso)
        add_to_list('artigos_aceitos', lista_bib_artigos_aceitos)
        add_to_list('apresentacoes_trabalhos', lista_bib_apresentacoes)
        add_to_list('textos_jornais', lista_bib_textos_jornais)
        add_to_list('outras_producoes', lista_bib_outras)

        # --- PRODUÇÃO TÉCNICA ---
        prod_tec = dados.get('producao_tecnica', {})
        def add_to_list_tec(chave, lista_destino):
            itens = prod_tec.get(chave, [])
            if itens:
                df_temp = pd.DataFrame(itens)
                df_temp['id_lattes'] = id_lattes
                lista_destino.append(df_temp)

        add_to_list_tec('softwares_com_patente', lista_tec_softwares_patente)
        add_to_list_tec('softwares_sem_patente', lista_tec_softwares_sem_patente)
        add_to_list_tec('produtos_tecnologicos', lista_tec_produtos)
        add_to_list_tec('processos_tecnicas', lista_tec_processos)
        add_to_list_tec('trabalhos_tecnicos', lista_tec_trabalhos)
        add_to_list_tec('outras_producoes_tecnicas', lista_tec_outras)
        add_to_list_tec('entrevistas', lista_tec_entrevistas)

        # --- PATENTES ---
        patentes = dados.get('patentes_registros', {})
        def add_to_list_pat(chave, lista_destino):
            itens = patentes.get(chave, [])
            if itens:
                df_temp = pd.DataFrame(itens)
                df_temp['id_lattes'] = id_lattes
                lista_destino.append(df_temp)

        add_to_list_pat('patentes', lista_pat_patentes)
        add_to_list_pat('programas_computador', lista_pat_programas)
        add_to_list_pat('desenhos_industriais', lista_pat_desenhos)


# ---------------------------------------------------------
# 3. CONSOLIDAÇÃO EM DATAFRAMES EXPLÍCITOS
# ---------------------------------------------------------
print("Consolidando DataFrames...")

def consolidar(lista):
    return pd.concat(lista, ignore_index=True) if lista else pd.DataFrame()

# DataFrames Gerais
df_pessoas = consolidar(lista_pessoas)
df_bancas = consolidar(lista_bancas)
df_eventos = consolidar(lista_eventos)
df_orientacoes = consolidar(lista_orientacoes)
df_premios = consolidar(lista_premios)
df_projetos = consolidar(lista_projetos)

# DataFrames Produção Bibliográfica
df_bib_artigos = consolidar(lista_bib_artigos)
df_bib_livros = consolidar(lista_bib_livros)
df_bib_capitulos = consolidar(lista_bib_capitulos)
df_bib_trab_congresso = consolidar(lista_bib_trabalhos_congresso)
df_bib_resumos_exp = consolidar(lista_bib_resumos_expandidos)
df_bib_resumos_cong = consolidar(lista_bib_resumos_congresso)
df_bib_art_aceitos = consolidar(lista_bib_artigos_aceitos)
df_bib_apresentacoes = consolidar(lista_bib_apresentacoes)
df_bib_textos_jornais = consolidar(lista_bib_textos_jornais)
df_bib_outras = consolidar(lista_bib_outras)

# DataFrames Produção Técnica
df_tec_soft_patente = consolidar(lista_tec_softwares_patente)
df_tec_soft_sem_patente = consolidar(lista_tec_softwares_sem_patente)
df_tec_produtos = consolidar(lista_tec_produtos)
df_tec_processos = consolidar(lista_tec_processos)
df_tec_trabalhos = consolidar(lista_tec_trabalhos)
df_tec_outras = consolidar(lista_tec_outras)
df_tec_entrevistas = consolidar(lista_tec_entrevistas)

# DataFrames Patentes
df_pat_patentes = consolidar(lista_pat_patentes)
df_pat_programas = consolidar(lista_pat_programas)
df_pat_desenhos = consolidar(lista_pat_desenhos)

Iniciando o processamento dos arquivos JSON...
Consolidando DataFrames...


# Tratando dados

## Informações pessoais

In [44]:
df_pessoas.info()

<class 'pandas.DataFrame'>
RangeIndex: 39 entries, 0 to 38
Data columns (total 11 columns):
 #   Column                 Non-Null Count  Dtype
---  ------                 --------------  -----
 0   id_lattes              39 non-null     str  
 1   nome_completo          39 non-null     str  
 2   nome_citacoes          39 non-null     str  
 3   sexo                   39 non-null     str  
 4   rotulo                 39 non-null     str  
 5   periodo                39 non-null     str  
 6   bolsa_produtividade    39 non-null     str  
 7   endereco_profissional  39 non-null     str  
 8   atualizacao_cv         39 non-null     str  
 9   url                    39 non-null     str  
 10  texto_resumo           39 non-null     str  
dtypes: str(11)
memory usage: 78.0 KB


In [45]:
# 1. Substituir strings vazias e espaços em branco por NaN
# Usa expressão regular para pegar "" ou "   "
df_pessoas.replace(r'^\s*$', np.nan, regex=True, inplace=True)

,id_lattes,nome_completo,nome_citacoes,sexo,rotulo,periodo,bolsa_produtividade,endereco_profissional,atualizacao_cv,url,texto_resumo
0,0211300683784278,Márcia Rosana Cerioli,"CERIOLI, M. R.;Cerioli, M;Cerioli, Marcia R.;C...",Masculino,* Sem rótulo,NaN,NaN,"Universidade Federal do Rio de Janeiro, Instit...",30/03/2026,http://lattes.cnpq.br/0211300683784278,Possui graduação em Matemática pela Universida...
1,9770406908381251,Claudio Luis de Amorim,"AMORIM, C. L.;AMORIM, CLAUDIO LUIS DE;AMORIM, ...",Masculino,* Sem rótulo,NaN,NaN,"Universidade Federal do Rio de Janeiro, Instit...",05/12/2024,http://lattes.cnpq.br/9770406908381251,Professor Titular do Programa de Engenharia de...
2,8154171198308578,Fábio Happ Botler,"BOTLER, F. H.;BOTLER, F.;BOTLER, FÁBIO;BOTLER,...",Masculino,* Sem rótulo,NaN,Nível 2,"Universidade de São Paulo, Instituto de Matemá...",12/03/2026,http://lattes.cnpq.br/8154171198308578,Possui graduação e mestrado em Matemática pela...
3,6243465206463403,Claudio Miceli de Farias,"FARIAS, C. M.;Farias, Claudio M.;Miceli, C.;MI...",Masculino,* Sem rótulo,NaN,NaN,"Universidade Federal do Rio de Janeiro, Instit...",30/07/2025,http://lattes.cnpq.br/6243465206463403,O professor Claudio Miceli de Farias fez gradu...
4,0523104569378276,Marcia Helena Costa Fampa,"FAMPA, M. H. C.;FAMPA, M.;FAMPA, MARCIA H.C.;F...",Masculino,* Sem rótulo,NaN,Nível 2,"Universidade Federal do Rio de Janeiro, Progra...",07/04/2026,http://lattes.cnpq.br/0523104569378276,Marcia é Professora da Universidade Federal do...
5,2002515486942024,Jayme Luiz Szwarcfiter,"SZWARCFITER, J. L.;Szwarcfiter, Jayme L.;Jayme...",Masculino,* Sem rótulo,NaN,Nível 1A (***,"Universidade Federal do Rio de Janeiro, COPPE ...",05/09/2025,http://lattes.cnpq.br/2002515486942024,Possui graduação em Engenharia Eletrônica pela...
6,9358511568098561,Edmundo Albuquerque de Souza e Silva,"de Souza e Silva, E.;de Souza e Silva, Edmundo...",Masculino,* Sem rótulo,NaN,Nível SR,"Universidade Federal do Rio de Janeiro, Instit...",15/01/2025,http://lattes.cnpq.br/9358511568098561,Edmundo de Souza e Silva é Engenheiro Elétrico...
7,5815607228657970,Henrique Luiz Cukierman,"CUKIERMAN, H. L.;CUKIERMAN, HENRIQUE LUIZ;CUKI...",Masculino,* Sem rótulo,NaN,NaN,"Universidade Federal do Rio de Janeiro, Instit...",08/11/2025,http://lattes.cnpq.br/5815607228657970,Possui graduação em Engenharia de Sistemas pel...
8,2704717555047499,Priscila Machado Vieira Lima,"LIMA, P. M. V.;LIMA, PRISCILA M. V.;LIMA, PRIS...",Masculino,* Sem rótulo,NaN,NaN,"Universidade Federal do Rio de Janeiro, Núcleo...",08/08/2025,http://lattes.cnpq.br/2704717555047499,Possui graduação em Informática pela Universid...
9,5349830056087028,Pedro Henrique González Silva,"González, P.H.;González, Pedro Henrique;GONZAL...",Masculino,* Sem rótulo,NaN,Nível C,"Universidade Federal do Rio de Janeiro, PESC -...",29/04/2026,http://lattes.cnpq.br/5349830056087028,Professor Adjunto na Universidade Federal do R...


In [46]:
# 2. Tratamento da Data de Atualização
# Converte a string '15/10/2025' para um tipo datetime
if 'atualizacao_cv' in df_pessoas.columns:
    df_pessoas['atualizacao_cv'] = pd.to_datetime(
        df_pessoas['atualizacao_cv'], 
        format='%d/%m/%Y', 
        errors='coerce' # Se tiver uma data bizarra (ex: 99/99/9999), vira nulo em vez de quebrar o script
    )

In [47]:
# 3. Limpeza do campo Rótulo
if 'rotulo' in df_pessoas.columns:
    # Remove o asterisco e espaços em branco nas pontas
    df_pessoas['rotulo'] = df_pessoas['rotulo'].str.replace('*', '', regex=False).str.strip()
    # Se o rótulo ficou "Sem rótulo", transforma em nulo verdadeiro
    df_pessoas['rotulo'] = df_pessoas['rotulo'].replace('Sem rótulo', np.nan)

In [48]:
# 5. Garantia de Tipagem da Chave Primária e Textos Longos
df_pessoas['id_lattes'] = df_pessoas['id_lattes'].astype(str)

In [49]:
if 'texto_resumo' in df_pessoas.columns:
    # Tira quebras de linha e espaços duplos extras no começo e no fim do resumo
    df_pessoas['texto_resumo'] = df_pessoas['texto_resumo'].str.strip()

In [50]:
if 'texto_resumo' in df_pessoas.columns:
    # Tira quebras de linha e espaços duplos extras no começo e no fim do resumo
    df_pessoas['texto_resumo'] = df_pessoas['texto_resumo'].str.strip()

In [51]:
df_pessoas.head()

,id_lattes,nome_completo,nome_citacoes,sexo,rotulo,periodo,bolsa_produtividade,endereco_profissional,atualizacao_cv,url,texto_resumo
0,0211300683784278,Márcia Rosana Cerioli,"CERIOLI, M. R.;Cerioli, M;Cerioli, Marcia R.;C...",Masculino,NaN,NaN,NaN,"Universidade Federal do Rio de Janeiro, Instit...",2026-03-30,http://lattes.cnpq.br/0211300683784278,Possui graduação em Matemática pela Universida...
1,9770406908381251,Claudio Luis de Amorim,"AMORIM, C. L.;AMORIM, CLAUDIO LUIS DE;AMORIM, ...",Masculino,NaN,NaN,NaN,"Universidade Federal do Rio de Janeiro, Instit...",2024-12-05,http://lattes.cnpq.br/9770406908381251,Professor Titular do Programa de Engenharia de...
2,8154171198308578,Fábio Happ Botler,"BOTLER, F. H.;BOTLER, F.;BOTLER, FÁBIO;BOTLER,...",Masculino,NaN,NaN,Nível 2,"Universidade de São Paulo, Instituto de Matemá...",2026-03-12,http://lattes.cnpq.br/8154171198308578,Possui graduação e mestrado em Matemática pela...
3,6243465206463403,Claudio Miceli de Farias,"FARIAS, C. M.;Farias, Claudio M.;Miceli, C.;MI...",Masculino,NaN,NaN,NaN,"Universidade Federal do Rio de Janeiro, Instit...",2025-07-30,http://lattes.cnpq.br/6243465206463403,O professor Claudio Miceli de Farias fez gradu...
4,0523104569378276,Marcia Helena Costa Fampa,"FAMPA, M. H. C.;FAMPA, M.;FAMPA, MARCIA H.C.;F...",Masculino,NaN,NaN,Nível 2,"Universidade Federal do Rio de Janeiro, Progra...",2026-04-07,http://lattes.cnpq.br/0523104569378276,Marcia é Professora da Universidade Federal do...


## Tratando orientações

In [52]:
df_orientacoes.head()

,titulo,ano_inicio,orientando,tipo_trabalho,instituicao,curso,id_lattes,status,nivel,ano_conclusao
0,Propriedades estruturais de grafos cordais e c...,2025,Rodrigo Fernandes Souto,Tese,Universidade Federal do Rio de Janeiro,,0211300683784278,em_andamento,doutorado,NaN
1,More on set graphs,2022,Bruno Bandeira Monteiro,Tese,Universidade Federal do Rio de Janeiro,,0211300683784278,em_andamento,doutorado,NaN
2,Colorações Seletivas em Grafos,2026,Rafael Paladini Meirelles,Dissertação,Universidade Federal do Rio de Janeiro,,0211300683784278,em_andamento,mestrado,NaN
3,Algoritmos de caminho em grafos,2025,Eduardo Naslausky,Dissertação,Universidade Federal do Rio de Janeiro,,0211300683784278,em_andamento,mestrado,NaN
4,Algoritmo de Caminho Mínimo: uma atividade de ...,2023,Caio de Campos,Trabalho de Conclusão de Curso,Universidade Federal do Rio de Janeiro,,0211300683784278,em_andamento,tcc,NaN


In [53]:
print(df_orientacoes['tipo_trabalho'].value_counts())
print(df_orientacoes['status'].value_counts())
print(df_orientacoes['nivel'].value_counts())
print(df_orientacoes['curso'].value_counts())

tipo_trabalho
Dissertação                       1273
Trabalho de Conclusão de Curso     696
Tese                               643
                                   195
Trabalho                            30
Monografia                          28
Iniciação científica                14
Name: count, dtype: int64
status
concluidas      2649
em_andamento     230
Name: count, dtype: int64
nivel
mestrado                1291
doutorado                648
tcc                      441
iniciacao_cientifica     345
pos_doutorado             81
especializacao            40
outros                    33
Name: count, dtype: int64
curso
    2879
Name: count, dtype: int64


In [54]:
df_orientacoes = df_orientacoes.rename(columns={'titulo': 'titulo_trabalho'})

In [55]:
import pandas as pd

print("Iniciando o tratamento da tabela de orientações...")

# 1. Tratamento de Strings: Remove espaços duplos e quebras de linha escondidas
colunas_texto = ['titulo_trabalho', 'orientando', 'tipo_trabalho', 'instituicao', 'curso']
for col in colunas_texto:
    df_orientacoes[col] = df_orientacoes[col].astype(str).str.strip()
    # Se, após limpar os espaços, o campo ficar vazio ou 'nan', preenche com 'Não informado'
    df_orientacoes[col] = df_orientacoes[col].replace({'': 'Não informado', 'nan': 'Não informado', 'None': 'Não informado'})

# 2. Conversão segura de anos (de float para Int64 com suporte a Nulo)
df_orientacoes['ano_conclusao'] = df_orientacoes['ano_conclusao'].astype('Int64')

# 3. Melhoria estética na coluna 'Nível' para os gráficos do Streamlit
mapeamento_nivel = {
    'mestrado': 'Mestrado',
    'doutorado': 'Doutorado',
    'tcc': 'TCC',
    'iniciacao_cientifica': 'Iniciação Científica',
    'pos_doutorado': 'Pós-Doutorado',
    'especializacao': 'Especialização',
    'outros': 'Outros'
}
df_orientacoes['nivel'] = df_orientacoes['nivel'].map(mapeamento_nivel).fillna(df_orientacoes['nivel'])

# 4. Melhoria estética na coluna 'Status' para os gráficos
mapeamento_status = {
    'concluidas': 'Concluída',
    'em_andamento': 'Em Andamento'
}
df_orientacoes['status'] = df_orientacoes['status'].map(mapeamento_status).fillna(df_orientacoes['status'])

print("\nTratamento concluído com sucesso! Verifique a nova estrutura:")
df_orientacoes.info()

print("\nAmostra dos dados tratados:")
display(df_orientacoes[['orientando', 'nivel', 'status', 'ano_conclusao']].head())

Iniciando o tratamento da tabela de orientações...

Tratamento concluído com sucesso! Verifique a nova estrutura:
<class 'pandas.DataFrame'>
RangeIndex: 2879 entries, 0 to 2878
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   titulo_trabalho  2879 non-null   str  
 1   ano_inicio       2879 non-null   int64
 2   orientando       2879 non-null   str  
 3   tipo_trabalho    2879 non-null   str  
 4   instituicao      2879 non-null   str  
 5   curso            2879 non-null   str  
 6   id_lattes        2879 non-null   str  
 7   status           2879 non-null   str  
 8   nivel            2879 non-null   str  
 9   ano_conclusao    2649 non-null   Int64
dtypes: Int64(1), int64(1), str(8)
memory usage: 809.0 KB

Amostra dos dados tratados:


,orientando,nivel,status,ano_conclusao
0,Rodrigo Fernandes Souto,Doutorado,Em Andamento,<NA>
1,Bruno Bandeira Monteiro,Doutorado,Em Andamento,<NA>
2,Rafael Paladini Meirelles,Mestrado,Em Andamento,<NA>
3,Eduardo Naslausky,Mestrado,Em Andamento,<NA>
4,Caio de Campos,TCC,Em Andamento,<NA>


## Informações acerca de periódicos publicados

In [56]:
df_bib_artigos.head()

,titulo,ano,autores,revista,volume,numero,paginas,issn,doi,qualis,id_lattes
0,On the (In)Dependence of the Peano Axioms for ...,2021,"CERIOLI, MÁRCIA R.; NOBREGA, HUGO ; SILVEIRA, ...",History and Philosophy of Logic,?,,1-19,1464-5149,http://dx.doi.org/10.1080/01445340.2021.1971005,,0211300683784278
1,Short proofs on the structure of general parti...,2021,"CERIOLI, MÁRCIA R.; MARTINS, TAÍSA",DISCRETE APPLIED MATHEMATICS,303,,8-13,0166-218X,http://dx.doi.org/10.1016/j.dam.2020.09.007,,0211300683784278
2,Transversals of longest paths,2020,"CERIOLI, MÁRCIA R.; FERNANDES, CRISTINA G. ; G...",DISCRETE MATHEMATICS,343,,111717,0012-365X,http://dx.doi.org/10.1016/j.disc.2019.111717,,0211300683784278
3,Intersection of longest paths in graph classes,2020,"CERIOLI, MÁRCIA R.; LIMA, PALOMA T.",DISCRETE APPLIED MATHEMATICS,281,,96-105,0166-218X,http://dx.doi.org/10.1016/j.dam.2019.03.022,,0211300683784278
4,On Edge-magic Labelings of Forests,2019,"CERIOLI, M. R.; FERNANDES, C. G. ; LEE, O. ; L...",ELECTRONIC NOTES IN THEORETICAL COMPUTER SCIENCE,346,,299-307,1571-0661,http://dx.doi.org/10.1016/j.entcs.2019.08.027,,0211300683784278


In [57]:
print("Aplicando tratamentos na tabela 'bib_artigos'...")

if not df_bib_artigos.empty:
    
    # 1. Transformar strings vazias ou só com espaços em nulos reais (NaN)
    df_bib_artigos.replace(r'^\s*$', np.nan, regex=True, inplace=True)
    
    # 2. Tratamento da Revista (Periódico)
    if 'revista' in df_bib_artigos.columns:
        # Força maiúsculo e remove espaços extras no início e no fim
        df_bib_artigos['revista'] = df_bib_artigos['revista'].str.upper().str.strip()

    # 4. Tratamento do Ano (Garantir que seja número inteiro)
    if 'ano' in df_bib_artigos.columns:
        # errors='coerce' transforma erros (ex: "Sem ano") em NaN
        # Int64 é o tipo inteiro do Pandas que aceita valores nulos
        df_bib_artigos['ano'] = pd.to_numeric(df_bib_artigos['ano'], errors='coerce').astype('Int64')

    # 5. Tratamento de Título, DOI e ISSN (Apenas remover espaços ocultos)
    colunas_texto = ['titulo', 'doi', 'issn', 'volume', 'numero', 'paginas']
    for col in colunas_texto:
        if col in df_bib_artigos.columns:
            df_bib_artigos[col] = df_bib_artigos[col].str.strip()

    # 6. Garantir tipagem da chave primária
    df_bib_artigos['id_lattes'] = df_bib_artigos['id_lattes'].astype(str)

print("Tratamento da tabela 'bib_artigos' concluído!")
display(df_bib_artigos[['ano', 'revista', 'doi', 'issn']].head())

Aplicando tratamentos na tabela 'bib_artigos'...
Tratamento da tabela 'bib_artigos' concluído!


,ano,revista,doi,issn
0,2021,HISTORY AND PHILOSOPHY OF LOGIC,http://dx.doi.org/10.1080/01445340.2021.1971005,1464-5149
1,2021,DISCRETE APPLIED MATHEMATICS,http://dx.doi.org/10.1016/j.dam.2020.09.007,0166-218X
2,2020,DISCRETE MATHEMATICS,http://dx.doi.org/10.1016/j.disc.2019.111717,0012-365X
3,2020,DISCRETE APPLIED MATHEMATICS,http://dx.doi.org/10.1016/j.dam.2019.03.022,0166-218X
4,2019,ELECTRONIC NOTES IN THEORETICAL COMPUTER SCIENCE,http://dx.doi.org/10.1016/j.entcs.2019.08.027,1571-0661


In [58]:
df_bib_artigos.info()

<class 'pandas.DataFrame'>
RangeIndex: 1997 entries, 0 to 1996
Data columns (total 11 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   titulo     1997 non-null   str  
 1   ano        1997 non-null   Int64
 2   autores    1997 non-null   str  
 3   revista    1997 non-null   str  
 4   volume     1960 non-null   str  
 5   numero     220 non-null    str  
 6   paginas    1980 non-null   str  
 7   issn       1956 non-null   str  
 8   doi        1516 non-null   str  
 9   qualis     0 non-null      str  
 10  id_lattes  1997 non-null   str  
dtypes: Int64(1), str(10)
memory usage: 666.6 KB


## Tratando informações de eventos

In [59]:
df_bib_trab_congresso.head()

,titulo,ano,doi,autores,evento,cidade,paginas,isbn,id_lattes
0,Another Calculational Proof of Cantor's Theorem,2022,http://dx.doi.org/10.5753/wbl.2022.223244,"CERIOLI, MÁRCIA R.; FREITAS, RENATA DE ; VIANA...",Workshop Brasileiro de Lógica,,9,,0211300683784278
1,Presenting Basic Graph Logic,2021,http://dx.doi.org/10.1007/978-3-030-86062-2,"CERIOLI, M. R.; SUGUITANI, L. ; VIANA, PETRUCIO",Diagrams,,132-148,,0211300683784278
2,Transversals of Longest Paths,2017,,"CERIOLI, M. R.; FERNANDES, C. G. ; GOMES, R. ;...",Latin and American Algorithms,,,,0211300683784278
3,On the (in)dependence of the Dedekind-Peano ax...,2017,http://dx.doi.org/10.5540/03.2017.005.01.0239,"CERIOLI, MA'RCIA; NOBREGA, HUGO ; SILVEIRA, GU...",CNMAC 2016 XXXVI Congresso Nacional de Matemát...,,,,0211300683784278
4,"L(2, 1)-coloração de k-árvores e grafos com tr...",2015,http://dx.doi.org/10.5540/03.2015.003.01.0241,"BARROS, GABRIEL F. ; POSNER, DANIEL F. D. ; CE...",XXXV CNMAC Congresso Nacional de Matemática Ap...,,,,0211300683784278


In [60]:
df_bib_trab_congresso.info()

<class 'pandas.DataFrame'>
RangeIndex: 3585 entries, 0 to 3584
Data columns (total 9 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   titulo     3585 non-null   str  
 1   ano        3585 non-null   int64
 2   doi        3585 non-null   str  
 3   autores    3585 non-null   str  
 4   evento     3585 non-null   str  
 5   cidade     3585 non-null   str  
 6   paginas    3585 non-null   str  
 7   isbn       3585 non-null   str  
 8   id_lattes  3585 non-null   str  
dtypes: int64(1), str(8)
memory usage: 1.0 MB


In [61]:
import numpy as np
import pandas as pd

print("Aplicando tratamentos na tabela 'df_bib_trab_congresso'...")

if not df_bib_trab_congresso.empty:
    
    # 1. Transformar strings vazias ou só com espaços em nulos reais (NaN)
    df_bib_trab_congresso.replace(r'^\s*$', np.nan, regex=True, inplace=True)
    
    # 2. Tratamento da coluna 'evento'
    if 'evento' in df_bib_trab_congresso.columns:
        df_bib_trab_congresso['evento'] = df_bib_trab_congresso['evento'].str.upper().str.strip()

    # 3. Tratamento da coluna 'ano'
    if 'ano' in df_bib_trab_congresso.columns:
        df_bib_trab_congresso['ano'] = pd.to_numeric(df_bib_trab_congresso['ano'], errors='coerce').astype('Int64')

    # 4. Tratamento de Título, DOI, ISBN e Paginas
    colunas_texto = ['titulo', 'doi', 'isbn', 'paginas']
    for col in colunas_texto:
        if col in df_bib_trab_congresso.columns:
            df_bib_trab_congresso[col] = df_bib_trab_congresso[col].str.strip()

    # 5. NOVO: Tratamento da coluna 'autores'
    if 'autores' in df_bib_trab_congresso.columns:
        # Remove espaços extras nas extremidades
        df_bib_trab_congresso['autores'] = df_bib_trab_congresso['autores'].str.strip()
        # Normaliza separadores: substitui ponto e vírgula por vírgula para manter padrão
        df_bib_trab_congresso['autores'] = df_bib_trab_congresso['autores'].str.replace(';', ',', regex=False)
        # Remove espaços duplos entre nomes/iniciais
        df_bib_trab_congresso['autores'] = df_bib_trab_congresso['autores'].str.replace(r'\s+', ' ', regex=True)
        # Opcional: Converter para maiúsculas para facilitar buscas
        df_bib_trab_congresso['autores'] = df_bib_trab_congresso['autores'].str.upper()

    # 6. Garantir tipagem da chave primária
    df_bib_trab_congresso['id_lattes'] = df_bib_trab_congresso['id_lattes'].astype(str)

print("Tratamento da tabela 'df_bib_trab_congresso' concluído!")
display(df_bib_trab_congresso[['ano', 'evento', 'autores']].head())

Aplicando tratamentos na tabela 'df_bib_trab_congresso'...
Tratamento da tabela 'df_bib_trab_congresso' concluído!


,ano,evento,autores
0,2022,WORKSHOP BRASILEIRO DE LÓGICA,"CERIOLI, MÁRCIA R., FREITAS, RENATA DE , VIANA..."
1,2021,DIAGRAMS,"CERIOLI, M. R., SUGUITANI, L. , VIANA, PETRUCIO"
2,2017,LATIN AND AMERICAN ALGORITHMS,"CERIOLI, M. R., FERNANDES, C. G. , GOMES, R. ,..."
3,2017,CNMAC 2016 XXXVI CONGRESSO NACIONAL DE MATEMÁT...,"CERIOLI, MA'RCIA, NOBREGA, HUGO , SILVEIRA, GU..."
4,2015,XXXV CNMAC CONGRESSO NACIONAL DE MATEMÁTICA AP...,"BARROS, GABRIEL F. , POSNER, DANIEL F. D. , CE..."


In [62]:
df_bib_trab_congresso.head(3)

,titulo,ano,doi,autores,evento,cidade,paginas,isbn,id_lattes
0,Another Calculational Proof of Cantor's Theorem,2022,http://dx.doi.org/10.5753/wbl.2022.223244,"CERIOLI, MÁRCIA R., FREITAS, RENATA DE , VIANA...",WORKSHOP BRASILEIRO DE LÓGICA,NaN,9,NaN,0211300683784278
1,Presenting Basic Graph Logic,2021,http://dx.doi.org/10.1007/978-3-030-86062-2,"CERIOLI, M. R., SUGUITANI, L. , VIANA, PETRUCIO",DIAGRAMS,NaN,132-148,NaN,0211300683784278
2,Transversals of Longest Paths,2017,NaN,"CERIOLI, M. R., FERNANDES, C. G. , GOMES, R. ,...",LATIN AND AMERICAN ALGORITHMS,NaN,NaN,NaN,0211300683784278


## Tratando informações de classificação

### Database de periódicos

In [63]:
import pandas as pd
import numpy as np

# ==========================================
# FUNÇÃO AUXILIAR DE LIMPEZA
# ==========================================
def formatar_issn(issn):
    # Converte para string, remove traços e tira espaços
    issn_str = str(issn).replace('-', '').strip()
    # Se for um valor nulo/vazio, retorna pd.NA para não virar "00000nan"
    if issn_str.lower() in ['nan', 'none', '', 'nat']:
        return pd.NA
    # Preenche com zeros à esquerda até completar 8 caracteres
    return issn_str.zfill(8)

print("Etapa 1: Carregando e preparando a base completa da Scopus...")
# Carrega o ficheiro original intacto da Scopus
df_scopus_raw = pd.read_excel('periodicos_percentil.xlsx')

# Limpeza de segurança padrão nas chaves de cruzamento 
df_scopus_raw['Title'] = df_scopus_raw['Title'].astype(str).str.upper().str.strip()

# Aplica a nova função de formatação de ISSN (8 dígitos)
df_scopus_raw['E-ISSN'] = df_scopus_raw['E-ISSN'].apply(formatar_issn)
df_scopus_raw['Print ISSN'] = df_scopus_raw['Print ISSN'].apply(formatar_issn)

# Mapeia TODOS os títulos/ISSNs que possuem alguma subárea de computação
mask_comput = df_scopus_raw['Scopus Sub-Subject Area'].str.contains('Comput', case=False, na=False)
titulos_computacao = set(df_scopus_raw[mask_comput]['Title'].unique())

# Unimos E-ISSN e Print ISSN no mesmo set para facilitar a checagem mais à frente
issns_computacao = set(df_scopus_raw[mask_comput]['E-ISSN'].dropna().unique()).union(
                   set(df_scopus_raw[mask_comput]['Print ISSN'].dropna().unique()))

# Ordena pelo Percentile (do maior para o menor) e remove duplicados de títulos.
df_scopus_unicos = df_scopus_raw.sort_values(by='Percentile', ascending=False)
df_scopus_unicos = df_scopus_unicos.drop_duplicates(subset=['Title'], keep='first').copy()

print("Etapa 2: Preparando a base de artigos do Lattes...")
# Limpeza de segurança padrão nas chaves dos artigos
df_bib_artigos['revista'] = df_bib_artigos['revista'].astype(str).str.upper().str.strip()

# Aplica a MESMA função no Lattes para garantir que ambos os lados tenham 8 dígitos no Match!
df_bib_artigos['issn'] = df_bib_artigos['issn'].apply(formatar_issn)

print("Etapa 3: Realizando o cruzamento exato inicial (ISSN e Nome Exato)...")
colunas_scopus = [
    'Scopus Source ID', 'Title', 'Percentile', 
    'Scopus ASJC Code (Sub-subject Area)', 'Scopus Sub-Subject Area', 'E-ISSN', 'Print ISSN'
]
df_scopus_filtro = df_scopus_unicos[colunas_scopus]

# Match pelo Nome Exato da Revista
df_match_nome = pd.merge(df_bib_artigos, df_scopus_filtro, left_on='revista', right_on='Title', how='inner')

# Match pelo E-ISSN
df_match_e_issn = pd.merge(df_bib_artigos, df_scopus_filtro, left_on='issn', right_on='E-ISSN', how='inner')

# Match pelo Print ISSN
df_match_print_issn = pd.merge(df_bib_artigos, df_scopus_filtro, left_on='issn', right_on='Print ISSN', how='inner')

# Consolida os sucessos exatos e remove possíveis duplicados
df_sucessos = pd.concat([df_match_nome, df_match_e_issn, df_match_print_issn], ignore_index=True)
df_sucessos = df_sucessos.drop_duplicates(subset=['titulo', 'id_lattes']).copy()

# Cria a coluna booleana de Computação
df_sucessos['Computation Area'] = (df_sucessos['Title'].isin(titulos_computacao) | 
                                   df_sucessos['E-ISSN'].isin(issns_computacao) | 
                                   df_sucessos['Print ISSN'].isin(issns_computacao))

print("Etapa 4: Busca Bidirecional (Nome Contido) para os artigos restantes...")
# Identifica quais artigos ainda não deram match
artigos_com_match_exato = set(df_sucessos['titulo'] + df_sucessos['id_lattes'])
df_restante = df_bib_artigos[~(df_bib_artigos['titulo'] + df_bib_artigos['id_lattes']).isin(artigos_com_match_exato)].copy()

# Prepara a lista do Scopus ordenada por tamanho do título para evitar roubo de match
df_scopus_filtro_sorted = df_scopus_filtro.copy()
df_scopus_filtro_sorted['tamanho_titulo'] = df_scopus_filtro_sorted['Title'].str.len()
df_scopus_filtro_sorted = df_scopus_filtro_sorted.sort_values(by='tamanho_titulo', ascending=False)
lista_scopus = df_scopus_filtro_sorted.to_dict('records')

# Função que busca um nome dentro do outro
def busca_bidirecional_revista(revista_lattes):
    if pd.isna(revista_lattes) or revista_lattes == 'NAN' or revista_lattes == '':
        return None

    for scopus in lista_scopus:
        titulo_scopus = scopus['Title']
        if pd.notna(titulo_scopus) and titulo_scopus != 'NAN' and titulo_scopus != "":
            # A mágica bidirecional ocorre aqui
            if (titulo_scopus in revista_lattes) or (revista_lattes in titulo_scopus):
                return scopus # Retorna o dicionário com os dados da Scopus encontrados
    return None

# Aplica a função iterativa apenas no dataframe restante (salva processamento)
resultados_parciais = df_restante['revista'].apply(busca_bidirecional_revista)

# Isola quem conseguiu match nessa etapa e os transforma em Dataframe
mask_encontrados = resultados_parciais.notna()
df_match_parcial = df_restante[mask_encontrados].copy()

if not df_match_parcial.empty:
    # Extrai as informações de Scopus do dicionário para as respectivas colunas
    dicts_encontrados = resultados_parciais[mask_encontrados]
    for col in colunas_scopus:
        df_match_parcial[col] = [d[col] for d in dicts_encontrados]
    
    # Aplica a verificação de computação
    df_match_parcial['Computation Area'] = (df_match_parcial['Title'].isin(titulos_computacao) | 
                                            df_match_parcial['E-ISSN'].isin(issns_computacao) | 
                                            df_match_parcial['Print ISSN'].isin(issns_computacao))
    
    # Junta esses novos achados à lista principal de sucessos
    df_sucessos = pd.concat([df_sucessos, df_match_parcial], ignore_index=True)
    df_sucessos = df_sucessos.drop_duplicates(subset=['titulo', 'id_lattes']).copy()

print("Etapa 5: Isolando e tratando as falhas definitivas...")
# Quem não passou nem no exato nem na busca parcial, cai aqui
df_falhas = df_restante[~mask_encontrados].copy()

# Preenche os atributos solicitados
df_falhas['Scopus Source ID'] = pd.NA
df_falhas['Title'] = pd.NA
df_falhas['Percentile'] = 0  
df_falhas['Scopus ASJC Code (Sub-subject Area)'] = pd.NA
df_falhas['Scopus Sub-Subject Area'] = pd.NA
df_falhas['E-ISSN'] = pd.NA
df_falhas['Print ISSN'] = pd.NA 
df_falhas['Computation Area'] = False  

print("Etapa 6: Consolidando a tabela final...")
# Empilha tudo (sucessos iniciais + sucessos bidirecionais + falhas)
df_artigos_final = pd.concat([df_sucessos, df_falhas], ignore_index=True)

# Garante a tipagem
df_artigos_final['Percentile'] = df_artigos_final['Percentile'].astype(int)
df_artigos_final['Computation Area'] = df_artigos_final['Computation Area'].astype(bool)

print("\n🚀 Tabela final construída com sucesso!")
print(f"Total de linhas: {len(df_artigos_final)}")

# Ordena as colunas finais
colunas_finais = [
    'id_lattes', 'titulo', 'revista', 'ano', # Dados do Artigo
    'Scopus Source ID', 'Title', 'Percentile', 
    'Scopus ASJC Code (Sub-subject Area)', 'Scopus Sub-Subject Area', 'E-ISSN', 'Print ISSN', 
    'Computation Area' 
]
df_artigos_final = df_artigos_final[colunas_finais]

# Mostra validação
display(df_artigos_final.sample(10))

Etapa 1: Carregando e preparando a base completa da Scopus...


Etapa 2: Preparando a base de artigos do Lattes...
Etapa 3: Realizando o cruzamento exato inicial (ISSN e Nome Exato)...
Etapa 4: Busca Bidirecional (Nome Contido) para os artigos restantes...
Etapa 5: Isolando e tratando as falhas definitivas...
Etapa 6: Consolidando a tabela final...

🚀 Tabela final construída com sucesso!
Total de linhas: 1990


,id_lattes,titulo,revista,ano,Scopus Source ID,Title,Percentile,Scopus ASJC Code (Sub-subject Area),Scopus Sub-Subject Area,E-ISSN,Print ISSN,Computation Area
981,8130520066599912,Improving Software Agent Communication with St...,INTERNATIONAL JOURNAL OF INFORMATION TECHNOLOG...,2010,11900154400,INTERNATIONAL JOURNAL OF INFORMATION TECHNOLOG...,64,1700,Computer Science (all),15541053,15541045,True
199,2002515486942024,Clique-Inverse Graphs of Bipartite Graphs,JOURNAL OF COMBINATORIAL MATHEMATICS AND COMBI...,2002,19700186819,JOURNAL OF COMBINATORIAL MATHEMATICS AND COMBI...,20,2600,Mathematics (all),NaN,08353026,False
545,2718664296804955,Serum Tryptase Monitoring in Indolent Systemic...,PLOS ONE,2013,10600153309,PLOS ONE,86,1000,Multidisciplinary,19326203,NaN,False
1218,5117568495536090,Fast relational learning using bottom clause p...,MACHINE LEARNING (DORDRECHT. ONLINE),2014,24775,MACHINE LEARNING,82,1712,Software,15730565,08856125,False
892,5370222318394867,On the absolute and relative oriented clique p...,PROCEDIA COMPUTER SCIENCE,2023,19700182801,PROCEDIA COMPUTER SCIENCE,62,1700,Computer Science (all),18770509,NaN,True
541,2718664296804955,A universal multilingual weightless neural net...,NEURAL NETWORKS,2017,24804,NEURAL NETWORKS,92,2805,Cognitive Neuroscience,18792782,08936080,False
797,4436183480921146,The one dimensional compartimentalised knapsac...,EUROPEAN JOURNAL OF OPERATIONAL RESEARCH,2007,22489,EUROPEAN JOURNAL OF OPERATIONAL RESEARCH,98,2611,Modeling and Simulation,NaN,03772217,True
46,8154171198308578,Decomposing 8-regular graphs into paths of len...,DISCRETE MATHEMATICS,2017,25892,DISCRETE MATHEMATICS,57,2607,Discrete Mathematics and Combinatorics,NaN,0012365X,True
421,3957046121364560,Canonical cuts of path powers,CONTRIBUTIONS TO DISCRETE MATHEMATICS,2024,21100297618,CONTRIBUTIONS TO DISCRETE MATHEMATICS,39,2607,Discrete Mathematics and Combinatorics,17150868,NaN,False
1318,2002515486942024,Improved Algorithms for Recognizing p-Helly an...,INFORMATION PROCESSING LETTERS (PRINT),2008,14388,INFORMATION PROCESSING LETTERS,41,1711,Signal Processing,NaN,00200190,True


In [64]:
# Cria a coluna indicando se o match ocorreu de forma adequada
# O método .notna() retorna True se existe um ID da Scopus, e False se for nulo (<NA>)
df_artigos_final['match_adequado'] = df_artigos_final['Scopus Source ID'].notna()

# Atualizando a lista de colunas para exibir essa nova no começo
colunas_finais = [
    'id_lattes', 'titulo', 'revista', 'ano', 'match_adequado', # <- Nova coluna aqui
    'Scopus Source ID', 'Title', 'Percentile', 
    'Scopus ASJC Code (Sub-subject Area)', 'Scopus Sub-Subject Area', 'E-ISSN', 
    'Computation Area'
]
df_artigos_final = df_artigos_final[colunas_finais]

print("\nColuna 'match_adequado' adicionada com sucesso!")
display(df_artigos_final[['titulo', 'revista', 'match_adequado', 'Percentile']].sample(10))


Coluna 'match_adequado' adicionada com sucesso!


,titulo,revista,match_adequado,Percentile
398,Message Delays for TDMA Scheme Under a Nonpree...,IEEE TRANSACTIONS ON COMMUNICATIONS,True,96
1548,A O(n) algorithm for projecting a vector on th...,RAIRO. OPERATIONS RESEARCH,True,66
1541,Acyclic orientations with path constraints,RAIRO. RECHERCHE OPÉRATIONNELLE,True,66
1830,Multiscale analysis for diffusion-driven neutr...,MATHEMATICAL AND COMPUTER MODELLING,True,79
218,Problemas de Roteamento de Veículos com Objeto...,JOURNAL OF THE BRAZILIAN COMPUTER SOCIETY,True,38
1189,A Sociotechnical Negotiation Mechanism to Supp...,CLEI ELECTRONIC JOURNAL,True,23
391,Node Mobility Modeling in Ad Hoc Networks thro...,LECTURE NOTES IN COMPUTER SCIENCE,True,43
878,Erratum to -Characterizing acyclic graphs by l...,DISCRETE APPLIED MATHEMATICS,True,73
703,Towards a framework to characterize ubiquiotus...,INFORMATION AND SOFTWARE TECHNOLOGY,True,89
1063,Applying data mining techniques for spatial di...,EXPERT SYSTEMS WITH APPLICATIONS,True,97


In [65]:
df_artigos_final.info()

<class 'pandas.DataFrame'>
RangeIndex: 1990 entries, 0 to 1989
Data columns (total 12 columns):
 #   Column                               Non-Null Count  Dtype 
---  ------                               --------------  ----- 
 0   id_lattes                            1990 non-null   str   
 1   titulo                               1990 non-null   str   
 2   revista                              1990 non-null   str   
 3   ano                                  1990 non-null   Int64 
 4   match_adequado                       1990 non-null   bool  
 5   Scopus Source ID                     1922 non-null   object
 6   Title                                1922 non-null   object
 7   Percentile                           1990 non-null   int64 
 8   Scopus ASJC Code (Sub-subject Area)  1922 non-null   object
 9   Scopus Sub-Subject Area              1922 non-null   object
 10  E-ISSN                               1176 non-null   object
 11  Computation Area                     1990 non-null   b

In [66]:
print("Renomeando as colunas do DataFrame final...")

# Dicionário com o mapeamento "Nome Antigo" : "Nome Novo"
mapeamento_colunas = {
    'titulo': 'titulo_artigo',
    'revista': 'titulo_revista_lattes',
    'ano': 'ano_pub',
    'Scopus Source ID': 'id_scopus',
    'Title': 'titulo_revista_scopus',
    'Percentile': 'maior_percentil',
    'Scopus ASJC Code (Sub-subject Area)': 'codigo_area_maior_percentil',
    'Scopus Sub-Subject Area': 'area_maior_percentil',
    'E-ISSN': 'issn',
    'Computation Area': 'computation_area'
}

# Aplica a renomeação diretamente no dataframe
df_artigos_final.rename(columns=mapeamento_colunas, inplace=True)

print("Colunas renomeadas com sucesso! Nova estrutura:")
df_artigos_final.info()

Renomeando as colunas do DataFrame final...
Colunas renomeadas com sucesso! Nova estrutura:
<class 'pandas.DataFrame'>
RangeIndex: 1990 entries, 0 to 1989
Data columns (total 12 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   id_lattes                    1990 non-null   str   
 1   titulo_artigo                1990 non-null   str   
 2   titulo_revista_lattes        1990 non-null   str   
 3   ano_pub                      1990 non-null   Int64 
 4   match_adequado               1990 non-null   bool  
 5   id_scopus                    1922 non-null   object
 6   titulo_revista_scopus        1922 non-null   object
 7   maior_percentil              1990 non-null   int64 
 8   codigo_area_maior_percentil  1922 non-null   object
 9   area_maior_percentil         1922 non-null   object
 10  issn                         1176 non-null   object
 11  computation_area             1990 non-null   bool  
dtypes: Int64(

### Database de conferências

In [67]:
df_google_raw = pd.read_csv('eventos_classificados_dois_idiomas.csv')

In [68]:
df_google_raw.head(3)

,Sigla,Nome do evento em inglês,Nome do evento,Estrato
0,AAAI,AAAI Conference on Artificial Intelligence,Conferência AAAI sobre Inteligência Artificial,A1
1,AAMAS,International Conference on Autonomous Agents ...,Conferência Internacional sobre Agentes Autôno...,A1
2,ACCV,Asian Conference on Computer Vision,Conferência Asiática sobre Visão Computacional,A1


In [69]:
df_google_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 781 entries, 0 to 780
Data columns (total 4 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   Sigla                     781 non-null    str  
 1   Nome do evento em inglês  781 non-null    str  
 2   Nome do evento            781 non-null    str  
 3   Estrato                   781 non-null    str  
dtypes: str(4)
memory usage: 124.6 KB


In [70]:
df_google_raw['Estrato'].value_counts()

Estrato
A3    171
A4    134
A1    110
B4     90
A2     86
B1     78
B2     60
B3     52
Name: count, dtype: int64

In [71]:
print("Substituindo as classificações na coluna 'Estrato'...")

# 1. Cria o dicionário de substituição ('Valor Antigo': 'Valor Novo')
mapeamento_estratos = {
    'B1': 'A5',
    'B2': 'A6',
    'B3': 'A7',
    'B4': 'A8'
}

# 2. Aplica a substituição apenas na coluna 'Estrato'
df_google_raw['Estrato'] = df_google_raw['Estrato'].replace(mapeamento_estratos)

# 3. Verifica o resultado para garantir que deu certo
print("\nNova distribuição de Estratos:")
display(df_google_raw['Estrato'].value_counts())

Substituindo as classificações na coluna 'Estrato'...

Nova distribuição de Estratos:


Estrato
A3    171
A4    134
A1    110
A8     90
A2     86
A5     78
A6     60
A7     52
Name: count, dtype: int64

In [72]:
# Transforma a coluna 'Nome do evento' para letras maiúsculas
df_google_raw['Nome do evento'] = df_google_raw['Nome do evento'].str.upper()

# Exibe as primeiras linhas para confirmar a alteração
display(df_google_raw.head())

,Sigla,Nome do evento em inglês,Nome do evento,Estrato
0,AAAI,AAAI Conference on Artificial Intelligence,CONFERÊNCIA AAAI SOBRE INTELIGÊNCIA ARTIFICIAL,A1
1,AAMAS,International Conference on Autonomous Agents ...,CONFERÊNCIA INTERNACIONAL SOBRE AGENTES AUTÔNO...,A1
2,ACCV,Asian Conference on Computer Vision,CONFERÊNCIA ASIÁTICA SOBRE VISÃO COMPUTACIONAL,A1
3,ACII,International Conference on Affective Computin...,CONFERÊNCIA INTERNACIONAL SOBRE COMPUTAÇÃO AFE...,A2
4,ACISP,Australasian Conference on Information Securit...,CONFERÊNCIA AUSTRALASIÁTICA SOBRE SEGURANÇA E ...,A4


In [73]:
import pandas as pd
import re

print("1. Preparando dados, removendo acentos e ordenando por tamanho do nome...")

# Cria uma função encadeada para limpar o texto: remove acentos, joga para maiúsculo e tira espaços nas bordas
def limpar_texto(serie):
    return (serie.astype(str)
            .str.normalize('NFKD')
            .str.encode('ascii', errors='ignore')
            .str.decode('utf-8')
            .str.upper()
            .str.replace(r'\s+', ' ', regex=True) # <- Escudo contra espaços múltiplos e invisíveis
            .str.strip())                     

# Aplica a limpeza nas bases
df_bib_trab_congresso['evento_limpo'] = limpar_texto(df_bib_trab_congresso['evento'])
df_google_raw['Nome do evento'] = limpar_texto(df_google_raw['Nome do evento'])
df_google_raw['Nome do evento em inglês'] = limpar_texto(df_google_raw['Nome do evento em inglês']) # Nova coluna
df_google_raw['Sigla'] = limpar_texto(df_google_raw['Sigla'])

# O truque de ouro atualizado: encontrar o maior nome entre PT e EN para ordenar
# Isso evita que um nome curto em uma das línguas roube o match de um nome longo na outra
df_google_raw['tamanho_pt'] = df_google_raw['Nome do evento'].str.len()
df_google_raw['tamanho_en'] = df_google_raw['Nome do evento em inglês'].str.len()
df_google_raw['tamanho_max'] = df_google_raw[['tamanho_pt', 'tamanho_en']].max(axis=1)

# Ordena pelo maior nome disponível
df_google_raw = df_google_raw.sort_values(by='tamanho_max', ascending=False)

# Transforma a base do Google em uma lista de dicionários para a busca ser ultrarrápida
lista_google = df_google_raw.to_dict('records')

print("2. Aplicando a lógica de match (Nomes PT/EN Bidirecional -> Sigla)...")

def encontrar_melhor_match(evento_lattes):
    if pd.isna(evento_lattes) or evento_lattes == 'NAN' or evento_lattes == '':
        return pd.NA, pd.NA, 'A8', 'Sem Match'

    # Tentativa 1: Verifica de forma BIDIRECIONAL em ambas as colunas (PT e EN)
    for google in lista_google:
        nome_pt = google['Nome do evento']
        nome_en = google['Nome do evento em inglês']
        
        # Checa a versão em Português
        if pd.notna(nome_pt) and nome_pt != 'NAN' and nome_pt != "":
            if (nome_pt in evento_lattes) or (evento_lattes in nome_pt):
                # Retorna sempre o nome em PT como principal para padronizar a tabela final
                return google['Sigla'], google['Nome do evento'], google['Estrato'], 'Por Nome PT'

        # Checa a versão em Inglês
        if pd.notna(nome_en) and nome_en != 'NAN' and nome_en != "":
            if (nome_en in evento_lattes) or (evento_lattes in nome_en):
                # Retorna sempre o nome em PT como principal para padronizar a tabela final
                return google['Sigla'], google['Nome do evento'], google['Estrato'], 'Por Nome EN'

    # Tentativa 2: Se falhar nos nomes, busca a Sigla protegida por limites de palavra (\b)
    for google in lista_google:
        sigla = google['Sigla']
        if pd.notna(sigla) and sigla != 'NAN' and sigla != "":
            padrao = r'\b' + re.escape(sigla) + r'\b'
            if re.search(padrao, evento_lattes):
                return google['Sigla'], google['Nome do evento'], google['Estrato'], 'Por Sigla'

    return pd.NA, pd.NA, 'A8', 'Sem Match'

print("Isso pode levar alguns segundos...")
# Aplica a função de busca
resultados = df_bib_trab_congresso['evento_limpo'].apply(encontrar_melhor_match)

print("3. Consolidando base final...")
df_artigos_congresso_final = df_bib_trab_congresso.copy()

# Extraindo os resultados da função para suas respectivas colunas
df_artigos_congresso_final['Sigla'] = [res[0] for res in resultados]
df_artigos_congresso_final['Nome do evento'] = [res[1] for res in resultados]
df_artigos_congresso_final['Estrato'] = [res[2] for res in resultados]
df_artigos_congresso_final['tipo_match'] = [res[3] for res in resultados]

# Remove colunas auxiliares de limpeza
df_artigos_congresso_final.drop(columns=['evento_limpo'], inplace=True)

# ==========================================
# RESUMO DOS RESULTADOS
# ==========================================
total_originais = len(df_artigos_congresso_final)
qtd_nome_pt = len(df_artigos_congresso_final[df_artigos_congresso_final['tipo_match'] == 'Por Nome PT'])
qtd_nome_en = len(df_artigos_congresso_final[df_artigos_congresso_final['tipo_match'] == 'Por Nome EN'])
qtd_sigla = len(df_artigos_congresso_final[df_artigos_congresso_final['tipo_match'] == 'Por Sigla'])
qtd_falhas = len(df_artigos_congresso_final[df_artigos_congresso_final['tipo_match'] == 'Sem Match'])

print(f"\n--- 📊 RELATÓRIO DE CRUZAMENTO DE EVENTOS ---")
print(f"Total de Artigos (Lattes): {total_originais}")
print(f"✅ Match por Nome PT: {qtd_nome_pt} ({round((qtd_nome_pt/total_originais)*100, 1)}%)")
print(f"✅ Match por Nome EN: {qtd_nome_en} ({round((qtd_nome_en/total_originais)*100, 1)}%)")
print(f"✅ Match por Sigla: {qtd_sigla} ({round((qtd_sigla/total_originais)*100, 1)}%)")
print(f"❌ Sem Match: {qtd_falhas} ({round((qtd_falhas/total_originais)*100, 1)}%)")

# Exibe uma amostra dos matches para validação
display(df_artigos_congresso_final[df_artigos_congresso_final['tipo_match'] != 'Sem Match'][['evento', 'Nome do evento', 'Estrato', 'tipo_match']].sample(5))

1. Preparando dados, removendo acentos e ordenando por tamanho do nome...
2. Aplicando a lógica de match (Nomes PT/EN Bidirecional -> Sigla)...
Isso pode levar alguns segundos...
3. Consolidando base final...

--- 📊 RELATÓRIO DE CRUZAMENTO DE EVENTOS ---
Total de Artigos (Lattes): 3585
✅ Match por Nome PT: 138 (3.8%)
✅ Match por Nome EN: 970 (27.1%)
✅ Match por Sigla: 807 (22.5%)
❌ Sem Match: 1670 (46.6%)


,evento,Nome do evento,Estrato,tipo_match
647,MIT SCALE LATIN AMERICA CONFERENCE,SIMPOSIO LATINO-AMERICANO DE INFORMATICA TEORICA,A3,Por Sigla
940,XIII SIMPÓSIO BRASILEIRO DE ENGENHARIA DE SOFT...,SIMPOSIO BRASILEIRO DE ENGENHARIA DE SOFTWARE,A3,Por Nome PT
2334,XLI SBPO-SIMPÓSIO BRASILEIRO DE PESQUISA OPERA...,BRAZILIAN SYMPOSIUM ON OPERATIONS RESEARCH,A4,Por Nome EN
3056,ECOOP,CONFERENCIA EUROPEIA SOBRE PROGRAMACAO ORIENTA...,A3,Por Sigla
3478,ACM SYMPOSIUM ON APPLIED COMPUTING,SIMPOSIO ACM SOBRE COMPUTACAO APLICADA,A1,Por Nome EN


In [74]:
import pandas as pd
import re
from rapidfuzz import fuzz

# [Seu Passo 1 (limpar_texto) continua exatamente igual aqui]

print("2. Aplicando a lógica Fuzzy de Alta Precisão (Nomes PT/EN -> Sigla)...")

# ==========================================
# CONFIGURAÇÃO DE RIGOR (0 a 100)
# ==========================================
# 90 é um valor excelente para evitar falsos positivos. 
# Só vai dar match se as palavras principais forem praticamente idênticas.
LIMIAR_CORTE_FUZZY = 95 

def encontrar_melhor_match_fuzzy(evento_lattes):
    if pd.isna(evento_lattes) or evento_lattes == 'NAN' or evento_lattes == '':
        return pd.NA, pd.NA, 'A8', 'Sem Match', 0

    melhor_google_match = None
    maior_score_encontrado = 0
    tipo_do_melhor_match = 'Sem Match'

    # ---------------------------------------------------------
    # TENTATIVA 1: Busca Exata pela Sigla (Altíssima Confiança)
    # ---------------------------------------------------------
    for google in lista_google:
        sigla = google['Sigla']
        if pd.notna(sigla) and sigla != 'NAN' and sigla != "":
            padrao = r'\b' + re.escape(sigla) + r'\b'
            if re.search(padrao, evento_lattes):
                # Se achou a sigla isolada, é match imediato (Score 100)
                return google['Sigla'], google['Nome do evento'], google['Estrato'], 'Por Sigla Exata', 100

    # ---------------------------------------------------------
    # TENTATIVA 2: Busca Fuzzy usando Token Set Ratio
    # ---------------------------------------------------------
    for google in lista_google:
        nome_pt = google['Nome do evento']
        nome_en = google['Nome do evento em inglês']
        
        score_pt = 0
        score_en = 0

        # Calcula a similaridade do conjunto de palavras em PT
        if pd.notna(nome_pt) and nome_pt != 'NAN' and nome_pt != "":
            score_pt = fuzz.token_set_ratio(evento_lattes, nome_pt)
            
        # Calcula a similaridade do conjunto de palavras em EN
        if pd.notna(nome_en) and nome_en != 'NAN' and nome_en != "":
            score_en = fuzz.token_set_ratio(evento_lattes, nome_en)

        # Pega o melhor score entre a versão PT e EN para este evento
        score_atual_max = max(score_pt, score_en)

        # Atualiza se encontrarmos um score melhor do que os anteriores
        if score_atual_max > maior_score_encontrado:
            maior_score_encontrado = score_atual_max
            melhor_google_match = google
            tipo_do_melhor_match = 'Fuzzy Nome PT' if score_pt >= score_en else 'Fuzzy Nome EN'

    # ---------------------------------------------------------
    # DECISÃO FINAL: O melhor match supera nosso limiar de segurança?
    # ---------------------------------------------------------
    if maior_score_encontrado >= LIMIAR_CORTE_FUZZY:
        return (
            melhor_google_match['Sigla'], 
            melhor_google_match['Nome do evento'], 
            melhor_google_match['Estrato'], 
            tipo_do_melhor_match, 
            maior_score_encontrado
        )

    # Se o melhor score foi menor que o limiar, titulo_evento_lattes, rejeitamos (evita falso positivo)
    return pd.NA, pd.NA, 'A8', 'Sem Match', maior_score_encontrado

print("Isso pode levar alguns segundos...")
# Aplica a função de busca
resultados = df_bib_trab_congresso['evento_limpo'].apply(encontrar_melhor_match_fuzzy)

print("3. Consolidando base final...")
df_artigos_congresso_final = df_bib_trab_congresso.copy()

# Extraindo os resultados, agora incluindo a coluna de Score para sua auditoria
df_artigos_congresso_final['Sigla'] = [res[0] for res in resultados]
df_artigos_congresso_final['Nome do evento'] = [res[1] for res in resultados]
df_artigos_congresso_final['Estrato'] = [res[2] for res in resultados]
df_artigos_congresso_final['tipo_match'] = [res[3] for res in resultados]
df_artigos_congresso_final['score_confianca'] = [res[4] for res in resultados]

df_artigos_congresso_final.drop(columns=['evento_limpo'], inplace=True)

2. Aplicando a lógica Fuzzy de Alta Precisão (Nomes PT/EN -> Sigla)...
Isso pode levar alguns segundos...
3. Consolidando base final...


In [ ]:
import pandas as pd

# 1. Definindo a "Zona Crítica"
# São os matches que passaram do limiar de corte, mas não são 100% idênticos
# Ajuste o nome do DataFrame para o seu de periódicos, se for o caso
limiar_inferior = 95  # O valor que definimos no código anterior
limiar_superior = 99  # Tudo abaixo de 100 (100 = match exato ou sigla perfeita)

df_zona_critica = df_artigos_congresso_final[
    (df_artigos_congresso_final['score_confianca'] >= limiar_inferior) & 
    (df_artigos_congresso_final['score_confianca'] <= limiar_superior)
].copy()

# 2. Ordenando pelo maior risco (scores mais baixos primeiro)
df_zona_critica = df_zona_critica.sort_values(by='score_confianca', ascending=True)

# 3. Selecionando apenas as colunas que importam para o "tira-teima" visual
# (Ajuste os nomes das colunas de acordo com a sua tabela de periódicos)
colunas_para_auditoria = [
    'evento',           # Nome original que veio do Lattes do pesquisador
    'Nome do evento',   # Nome que o algoritmo puxou da base de qualificação
    'Sigla',
    'Estrato', 
    'tipo_match',
    'score_confianca'
]

tabela_auditoria = df_zona_critica[colunas_para_auditoria]

# 4. Exibindo os resultados e aplicando estilo visual (se estiver no Jupyter/Colab)
print(f"⚠️ ATENÇÃO: Encontrados {len(tabela_auditoria)} registros na Zona Crítica (Score {limiar_inferior} a {limiar_superior}).")
print("Recomenda-se leitura atenta para garantir que não há homônimos.\n")

# Se você estiver usando Jupyter Notebook ou Google Colab, a linha abaixo 
# cria uma tabela com um gradiente de cores na coluna de score para facilitar a visão.
display(
    tabela_auditoria.style.background_gradient(
        subset=['score_confianca'], 
        cmap='YlOrRd_r', # Vermelho para 90, Amarelo para 99
        vmin=limiar_inferior, 
        vmax=limiar_superior
    )
)

⚠️ ATENÇÃO: Encontrados 123 registros na Zona Crítica (Score 95 a 99).
Recomenda-se leitura atenta para garantir que não há homônimos.



,evento,Nome do evento,Sigla,Estrato,tipo_match,score_confianca
414,XIX SIMPÓSIO BRASILEIRO DE TELECOMUNICAÇÕES,BRAZILIAN SYMPOSIUM ON TELECOMMUNICATIONS AND SIGNAL PROCESSING,SBRT,A8,Fuzzy Nome EN,95.121951
1073,XXI SIMPÓSIO BRASILEIRO DE TELECOMUNICAÇÕES,BRAZILIAN SYMPOSIUM ON TELECOMMUNICATIONS AND SIGNAL PROCESSING,SBRT,A8,Fuzzy Nome EN,95.121951
1074,XXI SIMPÓSIO BRASILEIRO DE TELECOMUNICAÇÕES,BRAZILIAN SYMPOSIUM ON TELECOMMUNICATIONS AND SIGNAL PROCESSING,SBRT,A8,Fuzzy Nome EN,95.121951
1080,XXI SIMPÓSIO BRASILEIRO DE TELECOMUNICAÇÕES,BRAZILIAN SYMPOSIUM ON TELECOMMUNICATIONS AND SIGNAL PROCESSING,SBRT,A8,Fuzzy Nome EN,95.121951
1081,XXI SIMPÓSIO BRASILEIRO DE TELECOMUNICAÇÕES,BRAZILIAN SYMPOSIUM ON TELECOMMUNICATIONS AND SIGNAL PROCESSING,SBRT,A8,Fuzzy Nome EN,95.121951
1082,XXI SIMPÓSIO BRASILEIRO DE TELECOMUNICAÇÕES,BRAZILIAN SYMPOSIUM ON TELECOMMUNICATIONS AND SIGNAL PROCESSING,SBRT,A8,Fuzzy Nome EN,95.121951
1083,XXI SIMPÓSIO BRASILEIRO DE TELECOMUNICAÇÕES,BRAZILIAN SYMPOSIUM ON TELECOMMUNICATIONS AND SIGNAL PROCESSING,SBRT,A8,Fuzzy Nome EN,95.121951
1062,XXI SIMPÓSIO BRASILEIRO DE TELECOMUNICAÇÕES,BRAZILIAN SYMPOSIUM ON TELECOMMUNICATIONS AND SIGNAL PROCESSING,SBRT,A8,Fuzzy Nome EN,95.121951
1071,XXI SIMPÓSIO BRASILEIRO DE TELECOMUNICAÇÕES,BRAZILIAN SYMPOSIUM ON TELECOMMUNICATIONS AND SIGNAL PROCESSING,SBRT,A8,Fuzzy Nome EN,95.121951
1347,XXI SIMPÓSIO BRASILEIRO DE TELECOMUNICAÇÕES,BRAZILIAN SYMPOSIUM ON TELECOMMUNICATIONS AND SIGNAL PROCESSING,SBRT,A8,Fuzzy Nome EN,95.121951


✅ Tabela exportada com sucesso para 'auditoria_zona_critica_fuzzy.xlsx'


In [76]:
df_artigos_congresso_final.head()
print(df_artigos_congresso_final.info())

<class 'pandas.DataFrame'>
RangeIndex: 3585 entries, 0 to 3584
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   titulo           3585 non-null   str    
 1   ano              3585 non-null   Int64  
 2   doi              981 non-null    str    
 3   autores          3585 non-null   str    
 4   evento           3573 non-null   str    
 5   cidade           0 non-null      str    
 6   paginas          2505 non-null   str    
 7   isbn             0 non-null      str    
 8   id_lattes        3585 non-null   str    
 9   Sigla            2098 non-null   str    
 10  Nome do evento   2098 non-null   str    
 11  Estrato          3585 non-null   str    
 12  tipo_match       3585 non-null   str    
 13  score_confianca  3585 non-null   float64
dtypes: Int64(1), float64(1), str(12)
memory usage: 1.3 MB
None


In [77]:
import pandas as pd

print("Padronizando os nomes das colunas de eventos...")

# Dicionário com o mapeamento "Nome Antigo" : "Nome Novo"
mapeamento_colunas_eventos = {
    'titulo': 'titulo_artigo',
    'evento': 'titulo_evento_lattes',
    'Sigla': 'sigla_evento_google',
    'Nome do evento': 'titulo_evento_google',
    'Estrato': 'estrato'
}

# Aplica a renomeação diretamente no dataframe
df_artigos_congresso_final.rename(columns=mapeamento_colunas_eventos, inplace=True)

print("Colunas renomeadas com sucesso! Nova estrutura:")
df_artigos_congresso_final.info()

Padronizando os nomes das colunas de eventos...
Colunas renomeadas com sucesso! Nova estrutura:
<class 'pandas.DataFrame'>
RangeIndex: 3585 entries, 0 to 3584
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   titulo_artigo         3585 non-null   str    
 1   ano                   3585 non-null   Int64  
 2   doi                   981 non-null    str    
 3   autores               3585 non-null   str    
 4   titulo_evento_lattes  3573 non-null   str    
 5   cidade                0 non-null      str    
 6   paginas               2505 non-null   str    
 7   isbn                  0 non-null      str    
 8   id_lattes             3585 non-null   str    
 9   sigla_evento_google   2098 non-null   str    
 10  titulo_evento_google  2098 non-null   str    
 11  estrato               3585 non-null   str    
 12  tipo_match            3585 non-null   str    
 13  score_confianca       3585 non-null   

In [78]:
print("Removendo as colunas 'cidade' e 'isbn'...")

# O parâmetro errors='ignore' é uma trava de segurança. 
# Se você rodar a célula duas vezes sem querer, ele não vai dar erro reclamando que a coluna já sumiu.
df_artigos_congresso_final.drop(columns=['cidade', 'isbn'], inplace=True, errors='ignore')

print("Colunas removidas com sucesso! Estrutura atualizada:")
df_artigos_congresso_final.info()

Removendo as colunas 'cidade' e 'isbn'...
Colunas removidas com sucesso! Estrutura atualizada:
<class 'pandas.DataFrame'>
RangeIndex: 3585 entries, 0 to 3584
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   titulo_artigo         3585 non-null   str    
 1   ano                   3585 non-null   Int64  
 2   doi                   981 non-null    str    
 3   autores               3585 non-null   str    
 4   titulo_evento_lattes  3573 non-null   str    
 5   paginas               2505 non-null   str    
 6   id_lattes             3585 non-null   str    
 7   sigla_evento_google   2098 non-null   str    
 8   titulo_evento_google  2098 non-null   str    
 9   estrato               3585 non-null   str    
 10  tipo_match            3585 non-null   str    
 11  score_confianca       3585 non-null   float64
dtypes: Int64(1), float64(1), str(10)
memory usage: 1.3 MB


# Conectando com DuckDB

In [79]:
import duckdb

print("Iniciando persistência no DuckDB com as tabelas de Artigos e Orientações...")
con = duckdb.connect('pesquisadores.duckdb')

# ==========================================
# 1. CRIAÇÃO DOS SCHEMAS E SEQUÊNCIAS
# ==========================================

# Tabela Mãe: Professores
query_cria_pessoas = """
CREATE TABLE IF NOT EXISTS tb_professores (
    id_lattes VARCHAR PRIMARY KEY,
    nome_completo VARCHAR,
    nome_citacoes VARCHAR,
    sexo VARCHAR,
    rotulo VARCHAR,
    periodo VARCHAR,
    bolsa_produtividade VARCHAR,
    endereco_profissional VARCHAR,
    atualizacao_cv TIMESTAMP,
    url VARCHAR,
    texto_resumo VARCHAR
);
"""
con.execute(query_cria_pessoas)

# Sequências para os IDs automáticos das três tabelas filhas
con.execute("CREATE SEQUENCE IF NOT EXISTS seq_id_artigo_periodico;")
con.execute("CREATE SEQUENCE IF NOT EXISTS seq_id_artigo_conferencia;")
con.execute("CREATE SEQUENCE IF NOT EXISTS seq_id_orientacao;")

# Tabela Filha 1: Artigos de Periódicos (Scopus)
query_cria_periodicos = """
CREATE TABLE IF NOT EXISTS tb_artigo_periodico (
    id_artigo_periodico INTEGER PRIMARY KEY DEFAULT nextval('seq_id_artigo_periodico'),
    id_lattes VARCHAR,
    titulo_artigo VARCHAR NOT NULL,
    titulo_revista_lattes VARCHAR,
    ano_pub INTEGER,
    match_adequado BOOLEAN,
    id_scopus VARCHAR,
    titulo_revista_scopus VARCHAR,
    maior_percentil INTEGER,
    codigo_area_maior_percentil VARCHAR,
    area_maior_percentil VARCHAR,
    issn VARCHAR,
    computation_area BOOLEAN,
    FOREIGN KEY (id_lattes) REFERENCES tb_professores(id_lattes)
);
"""
con.execute(query_cria_periodicos)

# Tabela Filha 2: Artigos de Conferências/Congressos (Google)
query_cria_conferencias = """
CREATE TABLE IF NOT EXISTS tb_artigo_conferencia (
    id_artigo_conferencia INTEGER PRIMARY KEY DEFAULT nextval('seq_id_artigo_conferencia'),
    id_lattes VARCHAR,
    titulo_artigo VARCHAR NOT NULL,
    ano INTEGER,
    doi VARCHAR,
    autores VARCHAR,
    titulo_evento_lattes VARCHAR,
    paginas VARCHAR,
    sigla_evento_google VARCHAR,
    titulo_evento_google VARCHAR,
    estrato VARCHAR,
    tipo_match VARCHAR,
    FOREIGN KEY (id_lattes) REFERENCES tb_professores(id_lattes)
);
"""
con.execute(query_cria_conferencias)

# Tabela Filha 3: Orientações (NOVA)
query_cria_orientacoes = """
CREATE TABLE IF NOT EXISTS tb_orientacoes (
    id_orientacao INTEGER PRIMARY KEY DEFAULT nextval('seq_id_orientacao'),
    id_lattes VARCHAR,
    titulo_trabalho VARCHAR,
    ano_inicio INTEGER,
    orientando VARCHAR,
    tipo_trabalho VARCHAR,
    instituicao VARCHAR,
    curso VARCHAR,
    status VARCHAR,
    nivel VARCHAR,
    ano_conclusao INTEGER,
    FOREIGN KEY (id_lattes) REFERENCES tb_professores(id_lattes)
);
"""
con.execute(query_cria_orientacoes)

print("Tabelas criadas com sucesso (ou já existentes).")

# ==========================================
# 2. INSERÇÃO DOS DADOS (Carga via Pandas)
# ==========================================
print("Limpando dados antigos (Filhas primeiro, Mãe depois)...")
# Apagar as 3 filhas antes da mãe para não violar a integridade relacional
con.execute("DELETE FROM tb_artigo_periodico")
con.execute("DELETE FROM tb_artigo_conferencia")
con.execute("DELETE FROM tb_orientacoes")
con.execute("DELETE FROM tb_professores")

print("Inserindo novos dados a partir dos DataFrames Pandas...")

# Inserção da Tabela Mãe (Professores)
if not df_pessoas.empty:
    con.execute("INSERT INTO tb_professores SELECT * FROM df_pessoas")

# Inserção da Tabela Filha 1 (Artigos Periódicos)
if not df_artigos_final.empty:
    con.execute("""
        INSERT INTO tb_artigo_periodico (
            id_lattes, titulo_artigo, titulo_revista_lattes, ano_pub, 
            match_adequado, id_scopus, titulo_revista_scopus, maior_percentil, 
            codigo_area_maior_percentil, area_maior_percentil, issn, computation_area
        )
        SELECT 
            id_lattes, titulo_artigo, titulo_revista_lattes, ano_pub, 
            match_adequado, id_scopus, titulo_revista_scopus, maior_percentil, 
            codigo_area_maior_percentil, area_maior_percentil, issn, computation_area 
        FROM df_artigos_final
    """)

# Inserção da Tabela Filha 2 (Artigos Conferência)
if not df_artigos_congresso_final.empty:
    con.execute("""
        INSERT INTO tb_artigo_conferencia (
            id_lattes, titulo_artigo, ano, doi, autores, 
            titulo_evento_lattes, paginas, sigla_evento_google, 
            titulo_evento_google, estrato, tipo_match
        )
        SELECT 
            id_lattes, titulo_artigo, ano, doi, autores, 
            titulo_evento_lattes, paginas, sigla_evento_google, 
            titulo_evento_google, estrato, tipo_match
        FROM df_artigos_congresso_final
    """)

# Inserção da Tabela Filha 3 (Orientações) - NOVA
if not df_orientacoes.empty:
    con.execute("""
        INSERT INTO tb_orientacoes (
            id_lattes, titulo_trabalho, ano_inicio, orientando, 
            tipo_trabalho, instituicao, curso, status, nivel, ano_conclusao
        )
        SELECT 
            id_lattes, titulo_trabalho, ano_inicio, orientando, 
            tipo_trabalho, instituicao, curso, status, nivel, ano_conclusao
        FROM df_orientacoes
    """)

con.close()
print("Processo finalizado! Banco 'pesquisadores.duckdb' atualizado com o schema completo.")

Iniciando persistência no DuckDB com as tabelas de Artigos e Orientações...
Tabelas criadas com sucesso (ou já existentes).
Limpando dados antigos (Filhas primeiro, Mãe depois)...
Inserindo novos dados a partir dos DataFrames Pandas...
Processo finalizado! Banco 'pesquisadores.duckdb' atualizado com o schema completo.
